# ДЗ 4. DML та DDL команди. Складні SQL-вирази: Olist Brazilian E-commerce

**Датасет:** Olist Brazilian E-commerce Public Dataset (Kaggle `olistbr/brazilian-ecommerce`). Близько 100 тис. замовлень за 2016–2018 роки.
**Використані таблиці (6):** `customers`, `orders`, `order_items`, `products`, `sellers`, `order_reviews`. Повний обсяг, без семплювання.

Шари даних:
- **raw** (`*_raw`) — дзеркало CSV, усі колонки як текст;
- **typed** (`olist_*`) — схема з PK / FK / CHECK / NOT NULL, на якій працюють усі запити;
- **службові** (`seller_alerts`, `seller_score`, `olist_customer_dim`, `customer_segments`) — результати DML і сегментації.

Головний принцип роботи: **перед кожною агрегацією контролюємо грануляність**. Таблиці мають різні рівні: позиція замовлення, замовлення, відгук, покупець. Зайвий JOIN множить рядки й спотворює метрики, і в розділі 2.2 це показано на числах.

## 1. Setup і typed-схема Olist

### 1.1. PostgreSQL через pgserver

In [1]:
# Install dependencies. PyPI ships pgserver wheels only for CPython 3.9-3.12, but its C extension
# is built against the stable ABI (abi3), so on Python 3.13+ we download the cp312 wheel and
# retag it as cp312-abi3, which pip accepts for any CPython >= 3.12.
import subprocess
import sys
from pathlib import Path

PIP = [sys.executable, "-m", "pip", "install", "-q"]
subprocess.run(PIP + ["psycopg2-binary", "sqlalchemy", "pandas", "kagglehub"], check=True)

if subprocess.run(PIP + ["pgserver"], capture_output=True).returncode != 0:
    wheel_dir = Path("/tmp/pgserver_wheel")
    subprocess.run(
        [sys.executable, "-m", "pip", "download", "-q", "pgserver==0.1.4",
         "--no-deps", "--only-binary=:all:", "--python-version", "3.12",
         "--platform", "manylinux2014_x86_64", "-d", str(wheel_dir)],
        check=True,
    )
    src = next(wheel_dir.glob("pgserver-0.1.4-cp312-cp312-*.whl"))
    dst = src.with_name(src.name.replace("cp312-cp312", "cp312-abi3"))
    src.rename(dst)
    subprocess.run(PIP + [str(dst)], check=True)

print(f"Python {sys.version.split()[0]}: залежності встановлено")

Python 3.13.13: залежності встановлено


In [2]:
import os

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 60)

pg = pgserver.get_server("/tmp/hw4_pg", cleanup_mode="stop")
# Pin psycopg2 explicitly: SQLAlchemy 2.1 maps "postgresql://" to psycopg 3, which is not installed
engine = create_engine(pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1), future=True)


def q(sql: str) -> pd.DataFrame:
    # Run a statement that returns rows (SELECT or DML ... RETURNING) and commit it
    with engine.begin() as conn:
        return pd.read_sql(text(sql), conn)


def run_sql(sql: str) -> None:
    # Run a multi-statement SQL script in one transaction
    with engine.begin() as conn:
        conn.exec_driver_sql(sql)


with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

/tmp/v13/lib/python3.13/site-packages/platformdirs/api.py:459: RuntimeDirWarning: XDG_RUNTIME_DIR is not set, falling back to /tmp/runtime-0
  return Path(self.user_runtime_dir)


PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


### 1.2. Отримання файлів Olist

Порядок джерел (перше доступне):
1. **`data/olist/`** у репозиторії — якщо CSV уже покладено поруч із notebook;
2. **KaggleHub** `olistbr/brazilian-ecommerce` (варіант A із завдання);
3. **офіційний GitHub-репозиторій Olist** [`olist/work-at-olist-data`](https://github.com/olist/work-at-olist-data/tree/master/datasets) — резерв без Kaggle credentials; файли ті самі.

Для контролю ідентичності між джерелами друкується MD5 кожного файлу.

In [3]:
import hashlib
from urllib.request import urlretrieve

FILES = {
    "olist_customers_raw": "olist_customers_dataset.csv",
    "olist_orders_raw": "olist_orders_dataset.csv",
    "olist_order_items_raw": "olist_order_items_dataset.csv",
    "olist_products_raw": "olist_products_dataset.csv",
    "olist_sellers_raw": "olist_sellers_dataset.csv",
    "olist_order_reviews_raw": "olist_order_reviews_dataset.csv",
}
LOCAL_DIR = Path("data/olist")
GITHUB_BASE = "https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets"


def has_all_files(directory: Path) -> bool:
    # True if every required CSV is present in the directory
    return all((directory / name).exists() for name in FILES.values())


def resolve_dataset_dir() -> tuple[Path, str]:
    # Return the directory with the Olist CSVs and a label of the source used
    if has_all_files(LOCAL_DIR):
        return LOCAL_DIR, "локальна папка data/olist"
    try:
        import kagglehub

        kaggle_dir = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
        if has_all_files(kaggle_dir):
            return kaggle_dir, "KaggleHub olistbr/brazilian-ecommerce"
    except Exception as exc:  # no credentials / no network: fall back to the Olist repository
        print(f"KaggleHub недоступний ({type(exc).__name__}), використовую GitHub Olist")
    LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    for name in FILES.values():
        urlretrieve(f"{GITHUB_BASE}/{name}", LOCAL_DIR / name)
    return LOCAL_DIR, "GitHub olist/work-at-olist-data"


dataset_dir, source_label = resolve_dataset_dir()
print(f"Джерело: {source_label}\nПапка: {dataset_dir}\n")
for name in FILES.values():
    path = dataset_dir / name
    print(f"{name:38s} {path.stat().st_size / 1e6:6.2f} MB  md5={hashlib.md5(path.read_bytes()).hexdigest()}")

Джерело: локальна папка data/olist
Папка: data/olist

olist_customers_dataset.csv              8.56 MB  md5=e585d546a08f221c310b85884fb74f4d
olist_orders_dataset.csv                17.65 MB  md5=8bd60e55c1ca229d9f70b62f3e72f22c
olist_order_items_dataset.csv           15.01 MB  md5=791f009968e979c9d1cffdd17146403d
olist_products_dataset.csv               2.38 MB  md5=e935401104a3669223b83c9a01fdf5ca
olist_sellers_dataset.csv                0.16 MB  md5=0a0949f31a8adb0914ba3e30486c52c8
olist_order_reviews_dataset.csv         14.45 MB  md5=914066be00d2db0c3d857f071cff1688


### 1.3. Raw-шар

CSV читаються з `dtype=str`. Raw-таблиця — текстове дзеркало файлу, без автоматичного вгадування типів pandas. Так zip-префікс `01037` не перетворюється на число `1037` ще до typed-шару, а цілі з пропусками не стають `float` (`40.0`). Порожні значення читаються як `NaN` і записуються в PostgreSQL як `NULL`.

In [4]:
raw_shapes = {}
for table_name, file_name in FILES.items():
    df = pd.read_csv(dataset_dir / file_name, dtype=str)
    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=10_000, method="multi")
    raw_shapes[table_name] = df.shape
    print(f"{table_name:26s} {df.shape}")

olist_customers_raw        (99441, 5)


olist_orders_raw           (99441, 8)


olist_order_items_raw      (112650, 7)


olist_products_raw         (32951, 9)


olist_sellers_raw          (3095, 4)


olist_order_reviews_raw    (99224, 7)


### 1.4. Typed-схема

DDL взято із завдання без змін структури. Єдина зміна — `DROP` для всіх службових таблиць винесено на початок, щоб Restart & Run All був відтворюваним незалежно від порядку попередніх запусків.

Ключові рішення:
- `olist_order_reviews` має технічний PK `review_row_id`, а `review_id` зберігається як джерельний атрибут: він **не унікальний** (див. 1.6);
- `olist_order_items` має складений PK `(order_id, order_item_id)`, тобто грануляність «позиція замовлення»;
- індекси `(seller_id, order_id)` і `reviews(order_id)` обслуговують seller-level агрегацію.

In [5]:
TYPED_DDL = r"""
DROP TABLE IF EXISTS customer_segments    CASCADE;
DROP TABLE IF EXISTS seller_score         CASCADE;
DROP TABLE IF EXISTS seller_alerts        CASCADE;
DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
DROP TABLE IF EXISTS olist_order_items    CASCADE;
DROP TABLE IF EXISTS olist_orders         CASCADE;
DROP TABLE IF EXISTS olist_customers      CASCADE;
DROP TABLE IF EXISTS olist_products       CASCADE;
DROP TABLE IF EXISTS olist_sellers        CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE INDEX idx_olist_customers_unique_id
    ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE INDEX idx_olist_order_items_seller_order
    ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id                TEXT,
    order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title     TEXT,
    review_comment_message   TEXT,
    review_creation_date     TIMESTAMP,
    review_answer_timestamp  TIMESTAMP
);

CREATE INDEX idx_olist_order_reviews_order_id
    ON olist_order_reviews (order_id);
"""
run_sql(TYPED_DDL)
print("typed-схему створено")

typed-схему створено


Перенесення raw → typed через `INSERT ... SELECT`. У raw-таблиці колонки з помилкою `lenght` звертаються за фактичною назвою з CSV.

In [6]:
LOAD_SQL = r"""
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
SELECT
    review_id,
    order_id,
    review_score::SMALLINT,
    review_comment_title,
    review_comment_message,
    review_creation_date::TIMESTAMP,
    review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;
"""
run_sql(LOAD_SQL)
print("typed-таблиці наповнено")

typed-таблиці наповнено


### 1.5. Smoke-test: raw vs typed

In [7]:
smoke = q("""
SELECT t.table_name, t.raw_rows, t.typed_rows, t.raw_rows - t.typed_rows AS lost_rows
FROM (
    SELECT 'customers' AS table_name,
           (SELECT COUNT(*) FROM olist_customers_raw) AS raw_rows,
           (SELECT COUNT(*) FROM olist_customers)     AS typed_rows
    UNION ALL SELECT 'orders',      (SELECT COUNT(*) FROM olist_orders_raw),        (SELECT COUNT(*) FROM olist_orders)
    UNION ALL SELECT 'order_items', (SELECT COUNT(*) FROM olist_order_items_raw),   (SELECT COUNT(*) FROM olist_order_items)
    UNION ALL SELECT 'products',    (SELECT COUNT(*) FROM olist_products_raw),      (SELECT COUNT(*) FROM olist_products)
    UNION ALL SELECT 'sellers',     (SELECT COUNT(*) FROM olist_sellers_raw),       (SELECT COUNT(*) FROM olist_sellers)
    UNION ALL SELECT 'reviews',     (SELECT COUNT(*) FROM olist_order_reviews_raw), (SELECT COUNT(*) FROM olist_order_reviews)
) AS t
ORDER BY t.table_name
""")
smoke

,table_name,raw_rows,typed_rows,lost_rows
0,customers,99441,99441,0
1,order_items,112650,112650,0
2,orders,99441,99441,0
3,products,32951,32951,0
4,reviews,99224,99224,0
5,sellers,3095,3095,0


In [8]:
assert (smoke.lost_rows == 0).all(), "typed-таблиця втратила рядки"
print("Усі рядки raw перенесено в typed")

Усі рядки raw перенесено в typed

**Висновок.** Усі шість typed-таблиць мають рівно стільки рядків, скільки raw: 99 441 клієнт і замовлення, 112 650 позицій, 32 951 товар, 3 095 продавців, 99 224 відгуки. Жоден рядок не відкинуто. Це водночас перевірка цілісності: кожен FK (`order_id` у позиціях і відгуках, `product_id`, `seller_id`) знайшов батьківський рядок, і жоден `CHECK` (статус, `review_score` 1–5, невід'ємні суми) не спрацював. Інакше `INSERT` завершився б помилкою.

### 1.6. Профіль якості та грануляності

Ці факти визначають рішення в усіх наступних розділах: чому `review_id` не може бути PK, де JOIN множить рядки і чому потрібен `customer_unique_id`.

In [9]:
profile = q("""
SELECT 'reviews: усього рядків' AS check_name, COUNT(*)::TEXT AS value FROM olist_order_reviews
UNION ALL SELECT 'reviews: унікальних review_id', COUNT(DISTINCT review_id)::TEXT FROM olist_order_reviews
UNION ALL SELECT 'reviews: review_id у кількох замовленнях',
       COUNT(*)::TEXT FROM (SELECT review_id FROM olist_order_reviews GROUP BY review_id HAVING COUNT(DISTINCT order_id) > 1) AS x
UNION ALL SELECT 'reviews: замовлень з > 1 review',
       COUNT(*)::TEXT FROM (SELECT order_id FROM olist_order_reviews GROUP BY order_id HAVING COUNT(*) > 1) AS x
UNION ALL SELECT 'items: замовлень з > 1 позицією',
       COUNT(*)::TEXT FROM (SELECT order_id FROM olist_order_items GROUP BY order_id HAVING COUNT(*) > 1) AS x
UNION ALL SELECT 'items: замовлень з > 1 продавцем',
       COUNT(*)::TEXT FROM (SELECT order_id FROM olist_order_items GROUP BY order_id HAVING COUNT(DISTINCT seller_id) > 1) AS x
UNION ALL SELECT 'customers: customer_id', COUNT(*)::TEXT FROM olist_customers
UNION ALL SELECT 'customers: customer_unique_id', COUNT(DISTINCT customer_unique_id)::TEXT FROM olist_customers
UNION ALL SELECT 'products: без категорії', COUNT(*)::TEXT FROM olist_products WHERE product_category_name IS NULL
UNION ALL SELECT 'orders: delivered без дати доставки',
       COUNT(*)::TEXT FROM olist_orders WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL
UNION ALL SELECT 'orders: період', MIN(order_purchase_timestamp)::DATE || ' … ' || MAX(order_purchase_timestamp)::DATE FROM olist_orders
""")
profile

,check_name,value
0,products: без категорії,610
1,customers: customer_id,99441
2,orders: delivered без дати доставки,8
3,reviews: усього рядків,99224
4,orders: період,2016-09-04 … 2018-10-17
5,reviews: замовлень з > 1 review,547
6,items: замовлень з > 1 позицією,9803
7,reviews: review_id у кількох замовленнях,789
8,customers: customer_unique_id,96096
9,items: замовлень з > 1 продавцем,1278


In [10]:
statuses = q("""
SELECT order_status, COUNT(*) AS n_orders,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
FROM olist_orders
GROUP BY order_status
ORDER BY n_orders DESC
""")
statuses

,order_status,n_orders,pct
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


**Висновок.**
- **`review_id` не є ключем:** 99 224 рядки, але лише 98 410 унікальних `review_id`. 789 з них стоять у кількох замовленнях, а 547 замовлень мають більше одного відгуку. PK на `review_id` зламав би завантаження, тому технічний `review_row_id` — необхідність, а не формальність.
- **Грануляність:** 9 803 замовлення мають кілька позицій, а 1 278 — кількох продавців. Саме ці замовлення спотворюють метрики при JOIN через `order_items` (2.2).
- **Покупці:** 99 441 `customer_id` відповідають лише 96 096 реальним покупцям (`customer_unique_id`).
- **Інші проблеми:** 610 товарів без категорії; 8 замовлень зі статусом `delivered`, але без дати доставки; статуси сильно незбалансовані (97,02 % `delivered`, `approved` і `created` — одиниці).

## 2. DML-запити з PostgreSQL-фічами

### 2.1. `INSERT ... RETURNING`: alert для продавця з найбільшою кількістю запізнень

`RETURNING` повертає щойно вставлений рядок з усіма згенерованими значеннями (`alert_id` з identity, `created_at` з `DEFAULT`) у тому самому запиті. Не потрібен окремий `SELECT ... ORDER BY alert_id DESC LIMIT 1`, який міг би повернути чужий рядок при паралельних записах. Результат одразу можна показати в notebook як audit-log операції.

**Ідемпотентність.** Запит із шаблону при кожному запуску додавав би новий дубль alert-у. Тому додано guard `NOT EXISTS`: якщо для цього продавця вже є alert `late_delivery`, повторний запуск нічого не вставляє і `RETURNING` повертає 0 рядків. Це показано другим запуском нижче.

Грануляність: `seller_orders` через `SELECT DISTINCT` зводить позиції до пар `seller × order`. Інакше замовлення з трьома позиціями одного продавця дало б три «запізнення».

In [11]:
run_sql("""
DROP TABLE IF EXISTS seller_alerts CASCADE;

CREATE TABLE seller_alerts (
    alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
    severity   SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details    TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
""")

ALERT_SQL = """
WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT
        so.seller_id,
        COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o
        ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO seller_alerts (seller_id, alert_type, severity, details)
SELECT
    ls.seller_id,
    'late_delivery',
    4,
    'Seller has ' || ls.late_orders || ' late delivered orders in the loaded dataset'
FROM late_sellers AS ls
WHERE NOT EXISTS (
    SELECT 1
    FROM seller_alerts AS a
    WHERE a.seller_id = ls.seller_id
      AND a.alert_type = 'late_delivery'
)
RETURNING alert_id, seller_id, alert_type, severity, details, created_at
"""
first_run = q(ALERT_SQL)
first_run

,alert_id,seller_id,alert_type,severity,details,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,Seller has 172 late delivered orders in the loaded dataset,2026-10-17 11:15:43.737275+00:00


In [12]:
second_run = q(ALERT_SQL)  # idempotency check: the guard must prevent a duplicate alert
print(f"Повторний запуск вставив рядків: {len(second_run)}")
q("SELECT COUNT(*) AS alerts_total FROM seller_alerts")

Повторний запуск вставив рядків: 0


,alerts_total
0,1


**Висновок.** Створено alert `alert_id = 1` для продавця `4a3ca931…` зі **172** запізнілими замовленнями. `RETURNING` одразу показав згенеровані `alert_id` і `created_at`, без окремого `SELECT`. Повторний запуск вставив **0** рядків, у таблиці так і лишився 1 alert. Guard `NOT EXISTS` зробив операцію ідемпотентною. Для конкурентного production-ETL надійніший варіант — частковий `UNIQUE`-індекс на `(seller_id, alert_type)` для відкритих alert-ів з `ON CONFLICT DO NOTHING`, бо `NOT EXISTS` не захищає від race condition між двома одночасними запусками.

### 2.2. `INSERT ... ON CONFLICT DO UPDATE`: idempotent upsert `seller_score`

- **Навіщо `SELECT DISTINCT` у `seller_orders`.** Позиції зводяться до грануляності `seller_id × order_id`. Замовлення, де продавець продав 3 товари, має рахуватися як **одне** замовлення з **одним** відгуком, а не три.
- **Навіщо агрегувати review до `order_id`.** У 547 замовлень більше одного відгуку (1.6). `review_per_order` усереднює їх в одну оцінку на замовлення. Тоді JOIN `seller_orders × review_per_order` має відношення 1:1 за `order_id`, і нічого не множиться.
- **Чому `ON CONFLICT (seller_id) DO UPDATE` повторюваний.** `seller_id` — PK. Перший запуск вставляє рядки, кожен наступний оновлює ті самі рядки замість вставки дублів: кількість рядків не змінюється, а значення перераховуються з поточних даних. Це показано подвійним запуском нижче.
- **Чому `avg_rating = NULL`, `n_reviews = 0`, `tier = NULL` — коректно.** Продавець, жодне замовлення якого не має відгуку, отримує `NULL` з `LEFT JOIN`. `AVG` по `NULL` дає `NULL`, а `COUNT(rpo.order_id)` рахує лише ненульові значення, тобто 0. Ставити 0 у рейтинг було б неправильно: «немає оцінки» ≠ «найгірша оцінка».

**Обмеження датасету.** Відгук в Olist прив'язаний до замовлення, а не до продавця. У 1 278 замовленнях кількох продавців (1.6) один order-level відгук зараховується кожному з них. Це прийнятне навчальне спрощення, але воно може несправедливо знижувати рейтинг продавця через чужу частину замовлення.

`RETURNING` тут повертає ~3 тис. рядків, тому його результат згорнуто через data-modifying CTE у зведення «скільки рядків у якому tier».

In [13]:
run_sql("""
DROP TABLE IF EXISTS seller_score CASCADE;

CREATE TABLE seller_score (
    seller_id   TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating  NUMERIC(3, 2),
    n_reviews   INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier        TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
    updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
""")

SELLER_SCORE_UPSERT = """
WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT
        so.seller_id,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
        COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
),
upserted AS (
    INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
    SELECT
        seller_id,
        avg_rating,
        n_reviews,
        CASE
            WHEN avg_rating IS NULL THEN NULL
            WHEN avg_rating >= 4.5 THEN 'gold'
            WHEN avg_rating >= 3.5 THEN 'silver'
            ELSE 'bronze'
        END AS tier
    FROM seller_rating
    ON CONFLICT (seller_id) DO UPDATE
    SET avg_rating = EXCLUDED.avg_rating,
        n_reviews  = EXCLUDED.n_reviews,
        tier       = EXCLUDED.tier,
        updated_at = NOW()
    RETURNING seller_id, avg_rating, n_reviews, tier, updated_at
)
SELECT
    COALESCE(tier, '(no reviews)') AS tier,
    COUNT(*) AS n_sellers,
    ROUND(AVG(avg_rating), 2) AS mean_avg_rating,
    SUM(n_reviews) AS total_reviewed_orders
FROM upserted
GROUP BY tier
ORDER BY n_sellers DESC
"""

upsert_run_1 = q(SELLER_SCORE_UPSERT)
upsert_run_1

,tier,n_sellers,mean_avg_rating,total_reviewed_orders
0,silver,1568,4.06,87123
1,gold,983,4.83,7093
2,bronze,539,2.30,5031
3,(no reviews),5,NaN,0


In [14]:
upsert_run_2 = q(SELLER_SCORE_UPSERT)  # second run: same rows updated, nothing duplicated
check = q("""
SELECT
    COUNT(*) AS rows_in_seller_score,
    COUNT(DISTINCT seller_id) AS distinct_sellers,
    (SELECT COUNT(DISTINCT seller_id) FROM olist_order_items) AS sellers_with_items
FROM seller_score
""")
assert check.rows_in_seller_score[0] == check.distinct_sellers[0] == check.sellers_with_items[0]
assert upsert_run_1.equals(upsert_run_2), "повторний upsert змінив результат"
check

,rows_in_seller_score,distinct_sellers,sellers_with_items
0,3095,3095,3095


**Доказ, що item-level join множить review score.** Той самий рейтинг, порахований «неправильно» (`order_items LEFT JOIN reviews` + `GROUP BY seller_id`), порівнюємо з `seller_score`.

In [15]:
inflation = q("""
WITH wrong AS (
    SELECT
        oi.seller_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS avg_rating_wrong,
        COUNT(rv.review_score) AS n_reviews_wrong
    FROM olist_order_items AS oi
    LEFT JOIN olist_order_reviews AS rv
        ON rv.order_id = oi.order_id
    GROUP BY oi.seller_id
)
SELECT
    SUM(w.n_reviews_wrong)                                       AS n_reviews_wrong_total,
    SUM(s.n_reviews)                                             AS n_reviews_correct_total,
    COUNT(*) FILTER (WHERE w.n_reviews_wrong <> s.n_reviews)     AS sellers_count_inflated,
    COUNT(*) FILTER (WHERE w.avg_rating_wrong IS DISTINCT FROM s.avg_rating) AS sellers_rating_changed,
    COUNT(*) FILTER (WHERE (w.avg_rating_wrong >= 4.5) IS DISTINCT FROM (s.avg_rating >= 4.5)) AS sellers_gold_status_changed,
    MAX(ABS(w.avg_rating_wrong - s.avg_rating))                  AS max_rating_shift
FROM wrong AS w
JOIN seller_score AS s
    ON s.seller_id = w.seller_id
""")
inflation

,n_reviews_wrong_total,n_reviews_correct_total,sellers_count_inflated,sellers_rating_changed,sellers_gold_status_changed,max_rating_shift
0,112372.0,99247,1458,1264,67,1.67


In [16]:
top_sellers = q("""
SELECT seller_id, avg_rating, n_reviews, tier
FROM seller_score
WHERE n_reviews >= 50
ORDER BY avg_rating DESC, n_reviews DESC
LIMIT 5
""")
top_sellers

,seller_id,avg_rating,n_reviews,tier
0,d13e50eaa47b4cbe9eb81465865d8cfc,4.82,67,gold
1,d9bd94811c3338dceb4181f3dbc0c73e,4.80,54,gold
2,d566c37fa119d5e66c4e9052e83ee4ea,4.69,64,gold
3,376a891762bbdecbc02b4b6adec3fdda,4.67,57,gold
4,1bb2bdb95f4841f1bba2c0d2cd83d3c9,4.64,80,gold


**Висновок.** `seller_score` містить **3 095** рядків, по одному на кожного продавця з позиціями. Повторний upsert не додав жодного рядка і дав ідентичне зведення (перевірено `assert`). Розподіл: silver — 1 568, gold — 983, bronze — 539. **5** продавців не мають жодного відгуку: у них `avg_rating = NULL`, `n_reviews = 0`, `tier = NULL`, як і очікувалося.

**Неправильний item-level join** дає 112 372 «відгуки» проти 99 247 коректних (+13 %). Кількість відгуків завищено в **1 458** продавців, рейтинг змінився в **1 264**. Для **67** продавців змінився б статус gold, а максимальне зміщення рейтингу становить **1,67** бала. Причина в тому, що відгук до замовлення з N позиціями одного продавця зараховується N разів. Тож продавець, який продає набори з кількох товарів, отримує надмірну вагу і своїх добрих, і своїх поганих оцінок.

Коректна сума 99 247 трохи перевищує кількість замовлень з відгуком (98 673). Різниця — це замовлення кількох продавців, де один order-level відгук зараховано кожному з них: те саме обмеження датасету, описане вище.

### 2.3. `UPDATE ... FROM`: прапорець запізнення в `olist_orders`

`ALTER TABLE ... ADD COLUMN IF NOT EXISTS` робить DDL повторюваним. `UPDATE ... FROM` бере обчислене значення з CTE за `order_id`.

Відносно шаблону додано одну умову: `WHERE o.is_late IS DISTINCT FROM f.is_late_calc`. Без неї запит **переписує всі ~99 тис. рядків** на кожному запуску (у PostgreSQL кожен `UPDATE` створює нову версію рядка, тобто bloat і зайвий WAL). З нею оновлюються лише рядки, де значення справді змінилося. Результат той самий, а ідемпотентність стає видимою: другий запуск оновлює **0** рядків.

`RETURNING` згорнуто через data-modifying CTE в audit-зведення.

In [17]:
run_sql("""
ALTER TABLE olist_orders
    ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;
""")

UPDATE_SQL = """
WITH delivery_flags AS (
    SELECT
        order_id,
        (
            order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE
        ) AS is_late_calc
    FROM olist_orders
),
updated AS (
    UPDATE olist_orders AS o
    SET is_late = f.is_late_calc
    FROM delivery_flags AS f
    WHERE f.order_id = o.order_id
      AND o.is_late IS DISTINCT FROM f.is_late_calc
    RETURNING o.order_id, o.customer_id, o.is_late
)
SELECT is_late, COUNT(*) AS n_rows_updated
FROM updated
GROUP BY is_late
"""

update_run_1 = q(UPDATE_SQL)
update_run_1

,is_late,n_rows_updated
0,True,6535


In [18]:
update_run_2 = q(UPDATE_SQL)  # second run must touch zero rows
print(f"Повторний UPDATE оновив рядків: {int(update_run_2.n_rows_updated.sum()) if len(update_run_2) else 0}")
q("""
SELECT
    order_status = 'delivered' AS is_delivered,
    is_late,
    COUNT(*) AS n_orders,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY order_status = 'delivered'), 2) AS pct_within_group
FROM olist_orders
GROUP BY order_status = 'delivered', is_late
ORDER BY is_delivered DESC, is_late
""")

Повторний UPDATE оновив рядків: 0


,is_delivered,is_late,n_orders,pct_within_group
0,True,False,89944,93.23
1,True,True,6534,6.77
2,False,False,2962,99.97
3,False,True,1,0.03


**Висновок.** Перший запуск позначив запізнілими **6 535** замовлень. `UPDATE` торкнувся лише цих рядків, бо решта вже мала `DEFAULT FALSE`. Другий запуск оновив **0** рядків. Серед доставлених запізнюються **6,77 %** (6 534 з 96 478). Ще одне запізніле замовлення має статус, відмінний від `delivered`. Перевірка в pandas показала 6 скасованих замовлень із заповненою датою доставки: це суперечність між статусом і часовими мітками, яку варто позначити як data-quality проблему.

### 2.4. `DELETE ... RETURNING`: контрольоване видалення застарілих alert-ів

Alert із 2.1 створено щойно, тож умову `created_at < NOW() - INTERVAL '30 days'` він не задовольняє, і перший `DELETE` поверне порожній результат. Щоб показати, що запит справді видаляє, спершу вставляємо **явно позначений тестовий** alert із `created_at` 45 днів тому (`DEFAULT` можна перевизначити явним значенням). `RETURNING` при видаленні — це audit-слід: видно, що саме зникло, і це можна зберегти в лог перед втратою даних.

In [19]:
DELETE_SQL = """
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at
"""

delete_before = q(DELETE_SQL)
print(f"(1) До вставки старого alert-у видалено рядків: {len(delete_before)}")

fixture = q("""
INSERT INTO seller_alerts (seller_id, alert_type, severity, details, created_at)
SELECT seller_id, 'data_quality', 1, 'TEST FIXTURE: backdated alert for DELETE demo',
       NOW() - INTERVAL '45 days'
FROM seller_alerts
ORDER BY alert_id
LIMIT 1
RETURNING alert_id, alert_type, details, created_at
""")
print("(2) Вставлено тестовий alert:")
display(fixture)

deleted = q(DELETE_SQL)
print("(3) DELETE ... RETURNING:")
display(deleted)

delete_again = q(DELETE_SQL)
print(f"(4) Повторний DELETE видалив рядків: {len(delete_again)}")
q("SELECT alert_id, seller_id, alert_type, created_at FROM seller_alerts ORDER BY alert_id")

(1) До вставки старого alert-у видалено рядків: 0
(2) Вставлено тестовий alert:


,alert_id,alert_type,details,created_at
0,2,data_quality,TEST FIXTURE: backdated alert for DELETE demo,2026-09-02 11:15:45.641488+00:00


(3) DELETE ... RETURNING:


,alert_id,seller_id,alert_type,created_at
0,2,4a3ca9315b744ce9f8e9374361493884,data_quality,2026-09-02 11:15:45.641488+00:00


(4) Повторний DELETE видалив рядків: 0


,alert_id,seller_id,alert_type,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,2026-10-17 11:15:43.737275+00:00


**Висновок.** Перший `DELETE` повернув **0** рядків, і це очікувано: єдиний alert створено щойно, він не старший за 30 днів. Після вставки тестового alert-у з датою 45 днів тому `DELETE ... RETURNING` видалив **рівно його** (`alert_id = 2`, `data_quality`) і показав, що саме зникло. Повторний `DELETE` знову повернув 0, а свіжий alert `late_delivery` залишився. Операція ідемпотентна: вона видаляє за станом даних, а не «N останніх рядків».

## 3. `customer_segments`

### 3.1. Дедуплікований customer dimension

`olist_customer_dim` має один рядок на `customer_unique_id`. `DISTINCT ON` з `ORDER BY ... order_purchase_timestamp DESC` бере атрибути з **останнього** замовлення покупця (актуальне місто чи штат, якщо покупець переїхав). Якщо час збігається, обираємо `customer_id ASC`, потім `order_id DESC`; той самий порядок використано у window-запиті нижче.

In [20]:
run_sql("""
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT REFERENCES olist_customers(customer_id),
    customer_state     CHAR(2),
    customer_city      TEXT
);

INSERT INTO olist_customer_dim (
    customer_unique_id,
    latest_customer_id,
    customer_state,
    customer_city
)
SELECT DISTINCT ON (c.customer_unique_id)
    c.customer_unique_id,
    c.customer_id AS latest_customer_id,
    c.customer_state,
    c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
ORDER BY
    c.customer_unique_id,
    o.order_purchase_timestamp DESC NULLS LAST,
    c.customer_id ASC,
    o.order_id DESC;
""")
q("""
SELECT
    (SELECT COUNT(*) FROM olist_customers)                     AS customer_ids,
    (SELECT COUNT(*) FROM olist_customer_dim)                  AS unique_customers,
    (SELECT COUNT(*) FROM (
        SELECT customer_unique_id FROM olist_customers
        GROUP BY customer_unique_id HAVING COUNT(*) > 1) AS x) AS customers_with_many_ids,
    (SELECT MAX(n) FROM (
        SELECT COUNT(*) AS n FROM olist_customers GROUP BY customer_unique_id) AS y) AS max_ids_per_customer
""")

,customer_ids,unique_customers,customers_with_many_ids,max_ids_per_customer
0,99441,96096,2997,17


### 3.2. DDL `customer_segments`

In [21]:
run_sql("""
CREATE TABLE customer_segments (
    segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    customer_unique_id  TEXT NOT NULL,
    segment_name        TEXT NOT NULL
        CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),

    rfm_score           SMALLINT NOT NULL
        CHECK (rfm_score BETWEEN 1 AND 5),

    recency_days        INTEGER
        CHECK (recency_days IS NULL OR recency_days >= 0),

    monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0
        CHECK (monetary_value >= 0),

    n_orders            INTEGER NOT NULL DEFAULT 0
        CHECK (n_orders >= 0),

    assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;
""")
q("""
SELECT
    conname,
    contype,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname
""")

,conname,contype,definition
0,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
1,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
2,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
3,customer_segments_pkey,p,PRIMARY KEY (segment_id)
4,customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_days >= 0)))
5,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
6,customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text, 'regular'..."
7,fk_customer_segments_customer_unique,f,FOREIGN KEY (customer_unique_id) REFERENCES olist_custom...


### 3.3. Наповнення через `INSERT ... SELECT ... ON CONFLICT`

`reference_date` — наступний день після останньої покупки в датасеті, а не `CURRENT_DATE` (пояснення в 3.4). `RETURNING` згорнуто в зведення за сегментами. Запит виконується двічі, щоб перевірити ідемпотентність.

In [22]:
SEGMENTS_UPSERT = """
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp
),
customer_stats AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
        MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c
        ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot
        ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT
        cs.customer_unique_id,
        cs.n_orders,
        cs.monetary_value,
        CASE
            WHEN cs.last_order_date IS NULL THEN NULL
            ELSE (rd.as_of_date - cs.last_order_date)::INTEGER
        END AS recency_days,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 'VIP'
            WHEN cs.last_order_date IS NOT NULL
                 AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                 AND cs.n_orders = 1 THEN 'new'
            WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
            ELSE 'inactive'
        END AS segment_name,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 5
            WHEN cs.monetary_value >= 500  THEN 4
            WHEN cs.n_orders >= 2          THEN 3
            WHEN cs.n_orders = 1           THEN 2
            ELSE 1
        END::SMALLINT AS rfm_score
    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
),
upserted AS (
    INSERT INTO customer_segments (
        customer_unique_id,
        segment_name,
        rfm_score,
        recency_days,
        monetary_value,
        n_orders
    )
    SELECT
        customer_unique_id,
        segment_name,
        rfm_score,
        recency_days,
        monetary_value,
        n_orders
    FROM segmented
    ON CONFLICT (customer_unique_id) DO UPDATE
    SET segment_name   = EXCLUDED.segment_name,
        rfm_score      = EXCLUDED.rfm_score,
        recency_days   = EXCLUDED.recency_days,
        monetary_value = EXCLUDED.monetary_value,
        n_orders       = EXCLUDED.n_orders,
        assigned_at    = NOW()
    RETURNING
        customer_unique_id,
        segment_name,
        rfm_score,
        recency_days,
        monetary_value,
        n_orders
)
SELECT
    segment_name,
    COUNT(*) AS n_customers,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_customers,
    ROUND(AVG(n_orders), 3) AS avg_orders,
    ROUND(AVG(monetary_value), 2) AS avg_monetary,
    ROUND(AVG(recency_days), 1) AS avg_recency_days
FROM upserted
GROUP BY segment_name
ORDER BY n_customers DESC
"""

segments_run_1 = q(SEGMENTS_UPSERT)
segments_run_1

,segment_name,n_customers,pct_customers,avg_orders,avg_monetary,avg_recency_days
0,regular,45992,47.86,1.070,224.61,307.4
1,inactive,40021,41.65,1.000,59.28,316.2
2,new,8891,9.25,1.000,140.40,72.4
3,VIP,1192,1.24,1.115,1591.48,290.1


In [23]:
segments_run_2 = q(SEGMENTS_UPSERT)
seg_check = q("""
SELECT
    COUNT(*)                            AS rows_in_segments,
    COUNT(DISTINCT customer_unique_id)  AS distinct_customers,
    (SELECT COUNT(*) FROM olist_customer_dim) AS customers_in_dim,
    MIN(segment_id) AS min_segment_id,
    MAX(segment_id) AS max_segment_id
FROM customer_segments
""")
assert seg_check.rows_in_segments[0] == seg_check.distinct_customers[0] == seg_check.customers_in_dim[0]
assert segments_run_1.equals(segments_run_2), "повторне наповнення змінило сегменти"
seg_check

,rows_in_segments,distinct_customers,customers_in_dim,min_segment_id,max_segment_id
0,96096,96096,96096,1,96096


Чому сегментація має бути саме на `customer_unique_id`: та сама метрика frequency, порахована на рівні `customer_id`.

In [24]:
grain_compare = q("""
SELECT
    'customer_id' AS grain,
    COUNT(*) AS n_entities,
    COUNT(*) FILTER (WHERE n_orders >= 2) AS repeat_buyers,
    MAX(n_orders) AS max_orders
FROM (
    SELECT customer_id, COUNT(*) AS n_orders
    FROM olist_orders
    GROUP BY customer_id
) AS by_customer_id
UNION ALL
SELECT
    'customer_unique_id',
    COUNT(*),
    COUNT(*) FILTER (WHERE n_orders >= 2),
    MAX(n_orders)
FROM customer_segments
""")
grain_compare

,grain,n_entities,repeat_buyers,max_orders
0,customer_id,99441,0,1
1,customer_unique_id,96096,2997,17


**Висновок.** Сегментовано **96 096** реальних покупців: regular — 47,86 %, inactive — 41,65 %, new — 9,25 %, VIP — 1,24 % (1 192 покупці із середніми витратами 1 591). Повторний запуск залишив 96 096 рядків, `segment_id` не зріс (1…96 096), а зведення за сегментами ідентичне. Повторний upsert оновлює `assigned_at`, тому повні рядки не є побайтово незмінними; стабільними залишаються ключі та бізнес-метрики.

Порівняння грануляностей показує, чому потрібен `customer_unique_id`. На рівні `customer_id` повторних покупців **0**, а максимум — 1 замовлення. На рівні `customer_unique_id` їх **2 997**, а максимум — **17** замовлень. Frequency на `customer_id` — константа, тобто марна ознака.

Обмеження сегментації з шаблону: `monetary_value` рахується з усіх замовлень, включно зі скасованими та недоставленими. Для production варто обмежитися `delivered`.

### 3.4. Пояснення

- **Чому `customer_unique_id`.** В Olist `customer_id` створюється для кожного замовлення (адреси доставки), тому на його рівні кожен «клієнт» має рівно одне замовлення. Порівняння вище це підтверджує: frequency вироджується в константу 1, повторних покупців немає. Реального покупця ідентифікує `customer_unique_id`. Лише на цьому рівні frequency і monetary мають сенс.
- **Навіщо `olist_customer_dim` для FK.** FK може посилатися лише на колонку з `PRIMARY KEY` або `UNIQUE`. В `olist_customers` значення `customer_unique_id` повторюються (одна людина — кілька `customer_id`), тож посилатися на нього неможливо. Dimension із `customer_unique_id` як PK дає цільову таблицю для FK і водночас фіксує грануляність «один рядок на покупця».
- **Чому `reference_date` з датасету, а не `CURRENT_DATE`.** Дані історичні (до 2018-10). З `CURRENT_DATE` recency для всіх була б ~3000 днів, сегмент `new` (≤ 90 днів) став би порожнім, а результат змінювався б щодня без зміни даних, і notebook не був би відтворюваним. `MAX(order_purchase_timestamp) + 1` — фіксована точка «станом на кінець датасету».
- **Як `ON CONFLICT` робить сегментацію повторюваною.** `UNIQUE (customer_unique_id)` — це арбітр конфлікту. Повторний запуск не вставляє другий рядок для того самого покупця, а оновлює наявний: `segment_id` (identity) зберігається, `assigned_at` оновлюється. Перевірка вище: кількість рядків = кількість покупців, `segment_id` не зростає, а зведення двох запусків ідентичні. При цьому `assigned_at` оновлюється, тому ідемпотентність тут означає відсутність дублікатів і стабільність бізнес-значень, а не незмінність кожного поля.

## 4. JOIN-запити з business-interpretation

### J1. INNER JOIN (4 таблиці)

**Business-question:** Які комбінації «категорія товару × штат продавця» дають найбільший revenue у доставлених замовленнях?

In [25]:
j1 = q("""
SELECT
    p.product_category_name,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON o.order_id = oi.order_id
JOIN olist_products AS p
    ON p.product_id = oi.product_id
JOIN olist_sellers AS s
    ON s.seller_id = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name, s.seller_state
ORDER BY revenue DESC NULLS LAST
LIMIT 10
""")
j1

,product_category_name,seller_state,n_orders,revenue
0,cama_mesa_banho,SP,8214,1070048.14
1,relogios_presentes,SP,4494,1016168.80
2,beleza_saude,SP,5684,791149.28
3,esporte_lazer,SP,4772,681832.26
4,moveis_decoracao,SP,4597,606384.62
5,cool_stuff,SP,2621,484196.53
6,utilidades_domesticas,SP,4021,472317.55
7,automotivo,SP,3018,438833.22
8,informatica_acessorios,SP,2783,388506.42
9,bebes,SP,1981,339129.74


**Interpretation:** Усі 10 найприбутковіших комбінацій мають `seller_state = SP`. Лідери — `cama_mesa_banho` (1,07 млн, 8 214 замовлень) і `relogios_presentes` (1,02 млн). Годинники й подарунки дають майже той самий revenue за майже вдвічі меншої кількості замовлень, тобто мають значно вищий чек. Сан-Паулу домінує як джерело пропозиції, тож seller-state як ознака сильно корелює з обсягом і потребує нормалізації. JOIN чотирьох таблиць тут безпечний: метрика рахується на рівні позиції, і кожна позиція має рівно один товар і одного продавця.

### J2. LEFT JOIN + COALESCE (feature engineering)

**Business-question:** Які customer-level features (кількість замовлень, витрати, середня оцінка, наявність відгуку) можна побудувати для реального покупця?

In [26]:
j2 = q("""
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_unique_id
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
customer_stats AS (
    SELECT
        ot.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score
    FROM order_totals AS ot
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
)
SELECT
    cd.customer_unique_id,
    cd.customer_state,
    COALESCE(cs.n_orders, 0) AS n_orders,
    COALESCE(cs.total_spend, 0)::NUMERIC(12, 2) AS total_spend,
    cs.avg_review_score,
    (cs.avg_review_score IS NOT NULL) AS has_review
FROM olist_customer_dim AS cd
LEFT JOIN customer_stats AS cs
    ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST
LIMIT 20
""")
j2

,customer_unique_id,customer_state,n_orders,total_spend,avg_review_score,has_review
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.0,True
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.0,True
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.0,True
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.0,True
4,459bef486812aa25204be022145caa62,ES,1,6922.21,NaN,False
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.0,True
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,1.0,True
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.0,True
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.0,True
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.0,True


**Interpretation:** Для кожного з 96 096 покупців отримано рядок ознак, включно з тими, хто не має позицій (`COALESCE(..., 0)` для `n_orders` і `total_spend`). Найбільший покупець витратив 13 664 і поставив оцінку **1,0**. Серед топ-20 за витратами **шестеро** поставили 1–2, тож високий чек не гарантує задоволення. CTE `review_per_order` зводить відгуки до одного рядка на замовлення ще до JOIN, тому оцінку не множать ні позиції, ні 547 замовлень з кількома відгуками.

Відступ від шаблону: `avg_review_score` **не** замінено на 0. Нуль поза шкалою 1–5 модель сприйняла б як «найгірша оцінка». Замість цього пропуск лишається `NULL`, а наявність відгуку подано окремою ознакою `has_review` (у рядку 5 — `False`).

### J3. FULL OUTER JOIN

**Business-question:** Які штати представлені тільки серед клієнтів, тільки серед продавців або в обох групах?

In [27]:
j3 = q("""
WITH customer_states AS (
    SELECT customer_state AS state, COUNT(*) AS n_customers
    FROM olist_customers
    GROUP BY customer_state
), seller_states AS (
    SELECT seller_state AS state, COUNT(*) AS n_sellers
    FROM olist_sellers
    GROUP BY seller_state
)
SELECT
    COALESCE(c.state, s.state) AS state,
    c.n_customers,
    s.n_sellers,
    CASE
        WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
        WHEN c.state IS NOT NULL THEN 'customers_only'
        ELSE 'sellers_only'
    END AS state_presence
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s
    ON s.state = c.state
ORDER BY state_presence DESC, state
""")
j3

,state,n_customers,n_sellers,state_presence
0,AL,413,NaN,customers_only
1,AP,68,NaN,customers_only
2,RR,46,NaN,customers_only
3,TO,280,NaN,customers_only
4,AC,81,1.0,both
5,AM,148,1.0,both
6,BA,3380,19.0,both
7,CE,1336,13.0,both
8,DF,2140,30.0,both
9,ES,2033,23.0,both


In [28]:
j3_extra = q("""
-- Cross-check: totals must equal the source table sizes (nothing lost on either side of the FULL OUTER JOIN)
SELECT
    (SELECT COUNT(*) FROM olist_customers) AS customers_total,
    (SELECT COUNT(*) FROM olist_sellers)   AS sellers_total
""")
j3_extra

,customers_total,sellers_total
0,99441,3095


**Interpretation:** З 27 штатів **23** мають і покупців, і продавців, а **4** (AL, AP, RR, TO) мають лише покупців. Штатів лише з продавцями **немає**. Пропозиція сильно сконцентрована: SP має 1 849 із 3 095 продавців (60 %), але лише 42 % покупців. У кількох штатах (AC, AM, MA, PA, PI) працює по одному продавцю. Для покупців у «customers_only» штатах кожна доставка міжрегіональна, а це кандидат на ознаку «відстань до продавця». FULL OUTER JOIN тут потрібен, бо звичайний JOIN мовчки втратив би ці 4 штати.

### J4. SELF JOIN

**Business-question:** Які пари продавців з одного штату, але з різних міст можуть бути кандидатами для регіонального порівняння, і скільки таких пар у кожному штаті?

In [29]:
j4 = q("""
SELECT
    s1.seller_id AS seller_a,
    s2.seller_id AS seller_b,
    s1.seller_state,
    s1.seller_city AS city_a,
    s2.seller_city AS city_b
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
   AND s1.seller_city IS DISTINCT FROM s2.seller_city
ORDER BY s1.seller_state, seller_a, seller_b
LIMIT 20
""")
j4

,seller_a,seller_b,seller_state,city_a,city_b
0,1444c08e64d55fb3c25f0f09c07ffcf2,2b402d5dc42554061f8ea98d1916f148,BA,lauro de freitas,irece
1,1444c08e64d55fb3c25f0f09c07ffcf2,398cb257329ef7af7f1943a8974a3cbc,BA,lauro de freitas,salvador
2,1444c08e64d55fb3c25f0f09c07ffcf2,4221a7df464f1fe2955934e30ff3a5a1,BA,lauro de freitas,bahia
3,1444c08e64d55fb3c25f0f09c07ffcf2,43753b27d77860f1654aa72e251a7878,BA,lauro de freitas,barro alto
4,1444c08e64d55fb3c25f0f09c07ffcf2,4aba391bc3b88717ce08eb11e44937b2,BA,lauro de freitas,arraial d'ajuda (porto seguro)
5,1444c08e64d55fb3c25f0f09c07ffcf2,4fb41dff7c50136976d1a5cf004a42e2,BA,lauro de freitas,feira de santana
6,1444c08e64d55fb3c25f0f09c07ffcf2,651530bf5c607240ccdd89a30c9c9712,BA,lauro de freitas,ipira
7,1444c08e64d55fb3c25f0f09c07ffcf2,659e8466eb3ff1b0e8740d74fb7bbedd,BA,lauro de freitas,eunapolis
8,1444c08e64d55fb3c25f0f09c07ffcf2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,lauro de freitas,salvador
9,1444c08e64d55fb3c25f0f09c07ffcf2,a3dd39f583bc80bd8c5901c95878921e,BA,lauro de freitas,salvador


In [30]:
j4_extra = q("""
-- Size of the self-join result by state: shows where the n*(n-1)/2 growth comes from
SELECT
    s1.seller_state,
    COUNT(*) AS n_pairs_diff_city
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
   AND s1.seller_city IS DISTINCT FROM s2.seller_city
GROUP BY s1.seller_state
ORDER BY n_pairs_diff_city DESC
LIMIT 5
""")
j4_extra

,seller_state,n_pairs_diff_city
0,SP,1457751
1,PR,51665
2,MG,27184
3,SC,17208
4,RJ,10151


**Interpretation:** SELF JOIN знаходить пари продавців з одного штату, але з різних міст. Умова `s1.seller_id < s2.seller_id` прибирає дзеркальні пари (A, B) / (B, A) і пари продавця із самим собою. `IS DISTINCT FROM` коректно порівнює міста навіть за наявності `NULL`. Розмір результату росте квадратично: лише SP дає **1 457 751** пару, бо пар порядку n(n−1)/2. Для регіонального бенчмаркінгу потрібно обмежувати кандидатів (одна категорія, схожий обсяг) ще до self-join. Вибірка також показує проблему якості в назвах міст: `bahia` — це назва штату, а не міста.

### J5. Anti-join (LEFT JOIN ... IS NULL)

**Business-question:** Які замовлення не мають жодного review-запису?

In [31]:
j5 = q("""
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp
FROM olist_orders AS o
LEFT JOIN olist_order_reviews AS rv
    ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL
ORDER BY o.order_purchase_timestamp
LIMIT 20
""")
j5

,order_id,customer_id,order_status,order_purchase_timestamp
0,cda873529ca7ab71f677d5ec11a40304,76c74aaff2f3f7355f46d9818ad092b8,shipped,2016-10-05 16:57:30
1,5da2004d85f2828349d887c989da2adc,95f592086150f0c95d3082aea047b816,delivered,2016-10-05 20:20:59
2,02190241f7190a1f3c7e0df95a749c6a,55c9044401ffab0c717f24405e2e042f,delivered,2016-10-07 10:52:22
3,0efd0bc268d34da3f01f4ff25a6d4335,95446917717bb58d553d107d0f1668f6,shipped,2016-10-07 15:53:31
4,c776863a93dc0740c6e7d78104b21413,1710b798ebdc5db6665c815b90a64999,delivered,2016-10-10 15:01:27
5,d730d8d0ea3149120703933fdf98b2d4,ce2a66464da8b80dedd7639a6e489a58,shipped,2017-01-13 20:12:41
6,05cd156f0f47579a36d7a52724fc31b8,976224a95faf328b17654b1b1411d5e1,delivered,2017-01-14 18:27:28
7,719b940ab2f1e351ffb14a0a02ffe3c4,ede9a1daf58d256c0d648ef7e1479010,delivered,2017-01-16 13:01:51
8,fbd82b1fe98967de0db29b6cf134d00a,45f5c9d984dbfa2105e3544cc85133bf,delivered,2017-01-18 08:55:36
9,171398a0cd843c65ac78aee9e05a1f49,f0244902c07b7c0799e468218579a925,delivered,2017-01-18 16:07:37


In [32]:
j5_extra = q("""
-- The same anti-join via NOT EXISTS, aggregated by order status
SELECT
    o.order_status,
    COUNT(*) AS orders_without_review,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_missing,
    COUNT(*) FILTER (WHERE NOT EXISTS (
        SELECT 1 FROM olist_order_items AS oi WHERE oi.order_id = o.order_id
    )) AS also_without_items
FROM olist_orders AS o
WHERE NOT EXISTS (
    SELECT 1
    FROM olist_order_reviews AS rv
    WHERE rv.order_id = o.order_id
)
GROUP BY o.order_status
ORDER BY orders_without_review DESC
""")
j5_extra

,order_status,orders_without_review,pct_of_missing,also_without_items
0,delivered,646,84.11,0
1,shipped,75,9.77,0
2,canceled,20,2.60,3
3,unavailable,14,1.82,14
4,processing,6,0.78,0
5,invoiced,5,0.65,0
6,created,2,0.26,2


**Interpretation:** Без жодного відгуку **768** замовлень (0,77 %). 84 % з них — `delivered` (646), тобто покупці просто не залишили оцінку. Решта — замовлення, що не дійшли до покупця (shipped, canceled, unavailable). У 19 із них немає навіть позицій. Обидва способи anti-join (`LEFT JOIN ... IS NULL` і `NOT EXISTS`) дають той самий набір. `NOT EXISTS` зручніший для агрегації, бо не створює проміжних `NULL`-рядків. Для ML це означає, що `has_review` — інформативна ознака, а не випадковий пропуск.

## 5. GROUP BY, HAVING і агрегати

### A1

**Business-question:** Які категорії товарів генерують понад 10 тис. revenue (без доставки) і яка в них середня ціна позиції?

In [33]:
a1 = q("""
SELECT
    p.product_category_name,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS gross_revenue,
    SUM(oi.freight_value)::NUMERIC(12, 2) AS freight,
    AVG(oi.price)::NUMERIC(8, 2) AS avg_item_price,
    ROUND(SUM(oi.freight_value) * 100.0 / NULLIF(SUM(oi.price), 0), 1) AS freight_pct_of_price
FROM olist_order_items AS oi
JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY p.product_category_name
HAVING SUM(oi.price) > 10000
ORDER BY gross_revenue DESC NULLS LAST
LIMIT 15
""")
a1

,product_category_name,n_orders,gross_revenue,freight,avg_item_price,freight_pct_of_price
0,beleza_saude,8836,1258681.34,182748.35,130.16,14.5
1,relogios_presentes,5624,1205005.68,100535.93,201.14,8.3
2,cama_mesa_banho,9417,1036988.68,204693.04,93.30,19.7
3,esporte_lazer,7720,988048.97,170274.23,114.34,17.2
4,informatica_acessorios,6689,911954.32,147318.08,116.51,16.2
5,moveis_decoracao,6449,729762.49,173483.29,87.56,23.8
6,cool_stuff,3632,635290.85,84039.10,167.36,13.2
7,utilidades_domesticas,5884,632248.66,147070.15,90.79,23.3
8,automotivo,3897,592720.11,92769.33,139.96,15.7
9,ferramentas_jardim,3518,485256.46,99171.18,111.63,20.4


**Interpretation:** За revenue без доставки лідирують `beleza_saude` (1,26 млн), `relogios_presentes` (1,21 млн) і `cama_mesa_banho` (1,04 млн). Профілі різні: годинники мають найвищу середню ціну позиції (201,14), а постільна білизна — низьку (93,30), зате найбільшу кількість замовлень (9 417). Частка доставки від ціни варіюється від 8,3 % (годинники) до 25 % (офісні меблі), тож для великогабаритних категорій логістика з'їдає помітну частину чека. `HAVING SUM(price) > 10000` відсіює довгий хвіст дрібних категорій.

### A2

**Business-question:** Який середній чек (AOV) у штатах покупців, де доставлено щонайменше 100 замовлень?

In [34]:
a2 = q("""
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_state,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY
        o.order_id,
        c.customer_state
)
SELECT
    customer_state,
    COUNT(*) AS n_orders,
    SUM(order_total)::NUMERIC(12, 2) AS gmv,
    (SUM(order_total) / NULLIF(COUNT(*), 0))::NUMERIC(8, 2) AS aov
FROM order_totals
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY aov DESC NULLS LAST
""")
a2

,customer_state,n_orders,gmv,aov
0,PB,517,137838.55,266.61
1,AL,397,94346.44,237.65
2,RO,243,56966.00,234.43
3,PA,946,212023.57,224.13
4,PI,476,105178.19,220.96
5,TO,274,60007.37,219.01
6,RN,474,100837.08,212.74
7,SE,335,70289.13,209.82
8,CE,1279,266593.21,208.44
9,MA,717,147803.55,206.14


**Interpretation:** Після фільтра «≥ 100 доставлених замовлень» лишилося 24 з 27 штатів. Найвищий AOV — на північному сході (PB 266,61, AL 237,65), найнижчий — у SP (142,51), хоча SP дає 40 501 замовлення й найбільший GMV (5,77 млн). Віддалені штати купують рідше, але дорожче, імовірно тому, що частину AOV становить вища вартість доставки від продавців із SP.

### A3

**Business-question:** Які категорії купують повторні покупці (≥ 2 замовлення) з найбільшими витратами?

In [35]:
a3 = q("""
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS total_spend,
    STRING_AGG(
        DISTINCT COALESCE(p.product_category_name, '(unknown)'),
        ', ' ORDER BY COALESCE(p.product_category_name, '(unknown)')
    ) AS categories
FROM olist_customers AS c
JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi
    ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 2
ORDER BY total_spend DESC NULLS LAST
LIMIT 15
""")
a3

,customer_unique_id,n_orders,total_spend,categories
0,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,eletroportateis
1,c8460e4251689ba205045f3ea17884a1,4,4080.00,telefonia
2,055ec572ac7f3c7bdd04a183830ebe59,2,3999.98,automotivo
3,59d66d72939bc9497e19d89c61a96d5f,2,3459.00,esporte_lazer
4,58c1b085b54c03a1f1ab5f13d64c2b1c,2,2999.98,pcs
5,d77aa95864ae5b42160937615628723a,2,2400.00,pcs
6,7b0eaf68a16e4808e5388c67345033c9,2,2238.42,bebes
7,86df00dc5fd68f4dd5d5945ca19f3ed6,3,2200.00,informatica_acessorios
8,6ddbc64bd04d40f7768ff088d94cbeb8,2,2108.00,eletrodomesticos
9,73601b1eec55943e90ce8d61253d5c09,2,2009.98,consoles_games


**Interpretation:** `STRING_AGG(DISTINCT ... ORDER BY ...)` збирає всі категорії покупця в один рядок. У 13 з 15 найбільших повторних покупців **одна** категорія: повторна покупка — це здебільшого «те саме ще раз» (телефонія, ПК, автотовари), а не крос-категорійний кошик. Для рекомендацій це означає, що сигнал «купував X» сильніший за сигнал «схожий покупець купував Y». `HAVING COUNT(DISTINCT order_id) >= 2` (у шаблоні було `>= 1`, а така умова нічого не фільтрує) відбирає лише повторних покупців.

### WHERE vs HAVING на прикладі A2

У A2 є обидва фільтри, і вони працюють на різних етапах:
- `WHERE o.order_status = 'delivered'` фільтрує **рядки до агрегації**: скасовані чи недоставлені замовлення взагалі не потрапляють у суми, тож не спотворюють AOV.
- `HAVING COUNT(*) >= 100` фільтрує **групи після агрегації**: відкидає штати, де замовлень замало для стабільного середнього. Цю умову неможливо записати в `WHERE`, бо `COUNT(*)` групи ще не існує на етапі фільтрації рядків.

Перевірка нижче показує, що перенесення умови в інше місце змінює результат.

In [36]:
where_having = q("""
WITH order_totals AS (
    SELECT o.order_id, o.order_status, c.customer_state
    FROM olist_orders AS o
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
    WHERE EXISTS (SELECT 1 FROM olist_order_items AS oi WHERE oi.order_id = o.order_id)
)
SELECT
    'WHERE delivered + HAVING >= 100' AS variant,
    COUNT(*) AS n_states,
    SUM(n) AS n_orders_used
FROM (
    SELECT customer_state, COUNT(*) AS n
    FROM order_totals
    WHERE order_status = 'delivered'
    GROUP BY customer_state
    HAVING COUNT(*) >= 100
) AS a
UNION ALL
SELECT
    'без WHERE, лише HAVING >= 100',
    COUNT(*),
    SUM(n)
FROM (
    SELECT customer_state, COUNT(*) AS n
    FROM order_totals
    GROUP BY customer_state
    HAVING COUNT(*) >= 100
) AS b
UNION ALL
SELECT
    'WHERE delivered, без HAVING',
    COUNT(*),
    SUM(n)
FROM (
    SELECT customer_state, COUNT(*) AS n
    FROM order_totals
    WHERE order_status = 'delivered'
    GROUP BY customer_state
) AS c
""")
where_having

,variant,n_states,n_orders_used
0,WHERE delivered + HAVING >= 100,24,96290.0
1,"без WHERE, лише HAVING >= 100",24,98471.0
2,"WHERE delivered, без HAVING",27,96478.0


**Висновок.** Три варіанти дають різні результати. Без `HAVING` у вибірці всі **27** штатів, з `HAVING >= 100` — **24** (вибувають AC, AP, RR). Без `WHERE` до сум потрапляють **98 471** замовлення замість **96 290**, тобто 2 181 недоставлене замовлення спотворило б AOV. Отже, `WHERE` змінює зміст метрики («що рахуємо»), а `HAVING` — набір груп («для кого показуємо»).

## 6. Set operations, window function і Reflection

### 6.1. `EXCEPT`: штати, де є покупці, але немає жодного продавця

**Чому `EXCEPT`.** Питання звучить як «множина A мінус множина B»: штати клієнтів, яких немає серед штатів продавців. `EXCEPT` формулює це прямо. Як і `UNION` та `INTERSECT`, він працює з **множинами** й прибирає дублікати, а нам потрібні унікальні коди штатів, а не 99 тис. повторів. Результат має збігатися з `customers_only` з FULL OUTER JOIN (J3): це незалежна перевірка двох різних способів.

In [37]:
s_except = q("""
SELECT customer_state AS state
FROM olist_customers

EXCEPT

SELECT seller_state AS state
FROM olist_sellers
ORDER BY state
""")
s_except

,state
0,AL
1,AP
2,RR
3,TO


In [38]:
# Cross-check: EXCEPT must give exactly the "customers_only" states of the FULL OUTER JOIN (J3)
full_outer_only = sorted(j3.loc[j3.state_presence == "customers_only", "state"])
assert sorted(s_except.state) == full_outer_only
print(f"EXCEPT = FULL OUTER JOIN customers_only: {full_outer_only}")

EXCEPT = FULL OUTER JOIN customers_only: ['AL', 'AP', 'RR', 'TO']


### 6.2. `INTERSECT`: штати, де є і покупці, і продавці

Для перетину множин потрібен саме `INTERSECT`. `UNION ALL` не підходить: він склеює набори з дублікатами і нічого не каже про спільні елементи.

In [39]:
s_intersect = q("""
SELECT customer_state AS state
FROM olist_customers

INTERSECT

SELECT seller_state AS state
FROM olist_sellers
ORDER BY state
""")
s_intersect

,state
0,AC
1,AM
2,BA
3,CE
4,DF
5,ES
6,GO
7,MA
8,MG
9,MS


**Висновок.** `EXCEPT` дає 4 штати (AL, AP, RR, TO), і це **точно** множина `customers_only` з FULL OUTER JOIN у J3. Два незалежні способи збігаються, що перевірено `assert`. `INTERSECT` дає 23 штати, що збігається з `both`. Разом 23 + 4 = 27, тобто всі штати покупців. Set operations тут лаконічніші за JOIN, коли потрібен лише склад множин, але FULL OUTER JOIN дає ще й кількості з обох боків.

### 6.3. Window function: найновіше замовлення кожного реального покупця (`ROW_NUMBER()`)

`ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY ... DESC)` нумерує замовлення в межах кожного покупця, від нового до старого. `rn = 1` — останнє замовлення. На відміну від `GROUP BY` + `MAX`, window function повертає **весь рядок** останнього замовлення (`order_id`, `customer_id`), а не лише максимальну дату. За однакового часу `customer_id ASC`, а потім `order_id DESC` задають детермінований порядок, ідентичний `olist_customer_dim`.

In [40]:
w_latest = q("""
WITH ranked_orders AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        o.order_id,
        o.order_purchase_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp DESC, o.customer_id ASC, o.order_id DESC
        ) AS rn,
        COUNT(*) OVER (PARTITION BY c.customer_unique_id) AS n_orders_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    customer_unique_id,
    customer_id,
    order_id,
    order_purchase_timestamp,
    n_orders_total
FROM ranked_orders
WHERE rn = 1
ORDER BY n_orders_total DESC, customer_unique_id
LIMIT 10
""")
w_latest

,customer_unique_id,customer_id,order_id,order_purchase_timestamp,n_orders_total
0,8d50f5eadf50201ccdcedfb9e2ac8455,a682769c4bc10fc6ef2101337a6c83c9,d3582fd5ccccd9cb229a63dfb417c86f,2018-08-20 19:14:26,17
1,3e43e6105506432c953e165fb2acf44c,1815fdaaae692479110428598596b5e5,ff89ef7b3952bba5ac06d61c4a79ffbe,2018-02-27 18:36:39,9
2,1b6c7548a2a1f9037c1fd3ddfed95f33,fff7466a253c0e59499ea943462c10f9,43f08913407cac4e74a898d968e58c1a,2018-02-14 13:22:12,7
3,6469f99c1f9dfae7733b25662e7f1782,4478700ff008d3fffa02e95d89b9795f,ccb1046449f1a494de800567e7cb5979,2018-06-28 00:43:34,7
4,ca77025e7201e3b30c44b472ff346268,6ccedfba5919d72fcc8c51bfa982de62,8dacd1da7c4dc40d5c59e7ddd93b3ac8,2018-06-01 11:38:29,7
5,12f5d6e1cbf93dafd9dcc19095df0b3d,ed1793d2d1e4175d5846ce7ebb4a01f5,747996a66f5aa711deb8ae58f5ae46a0,2017-01-05 15:25:10,6
6,47c1a3033b8b77b3ab6e109eb4d5fdf3,5daf8c734e89f88e5305f9283e7ca135,b86ec88f5bffbf1f61fb5daed332e355,2018-01-24 15:15:26,6
7,63cfc61cee11cbe306bff5857d00bfe4,906614004e38a1e922ff033563bfdcf2,c29713133e5e33937af1a6e1e1370552,2018-05-28 17:20:02,6
8,dc813062e0fc23409cd255f7f53c7074,4fd0b9bb860c9a3d841746e1cb742314,f0031384d36ff045ee1e866942d0fa7a,2018-08-23 00:07:26,6
9,de34b16117594161a6a89c50b289d35a,cbbd508281ea66e2f36b4acfc28a85b0,d29eca3633b0b60adf268cae9133cd93,2018-01-17 23:34:21,6


Перевірка узгодженості: `latest_customer_id` в `olist_customer_dim` (через `DISTINCT ON`) має збігатися з `customer_id` останнього замовлення за `ROW_NUMBER()` для всіх покупців.

In [41]:
w_check = q("""
WITH ranked_orders AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp DESC, o.customer_id ASC, o.order_id DESC
        ) AS rn
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    COUNT(*) AS customers_compared,
    COUNT(*) FILTER (WHERE r.customer_id = cd.latest_customer_id) AS same_latest,
    COUNT(*) FILTER (WHERE r.customer_id <> cd.latest_customer_id) AS different_latest
FROM ranked_orders AS r
JOIN olist_customer_dim AS cd
    ON cd.customer_unique_id = r.customer_unique_id
WHERE r.rn = 1
""")
assert int(w_check.loc[0, "different_latest"]) == 0, "Різні правила вибору останнього замовлення"
w_check

,customers_compared,same_latest,different_latest
0,96096,96096,0


**Interpretation:** `ROW_NUMBER()` повертає повний рядок останнього замовлення для кожного з 96 096 покупців. Найактивніший покупець має **17** замовлень, останнє — 2018-08-20. `COUNT(*) OVER (PARTITION BY ...)` додає загальну кількість замовлень у той самий рядок без окремого `GROUP BY` і JOIN.

Перехресна перевірка з `olist_customer_dim` показала **0** розбіжностей: обидва запити вибрали той самий `customer_id` для всіх **96 096** покупців. У 269 покупців кілька замовлень мають однаковий останній timestamp до секунди; однаковий tie-breaker (`customer_id ASC`, далі `order_id DESC`) усуває неоднозначність між dimension і window-запитом. Без цього правила попередні варіанти розходилися для 135 покупців.

### 6.4. Reflection

**Найскладніший JOIN.** Найскладнішим виявився не синтаксичний FULL OUTER, а LEFT JOIN для features. Тут треба було одночасно зберегти покупців без замовлень, не помножити відгуки й не підмінити відсутню оцінку нулем. Найважче було сформулювати як business-question self-join: технічно він тривіальний, але дає 1,46 млн пар лише для SP і без додаткових обмежень не відповідає на жодне практичне питання.

**Найкорисніша фіча для ETL** — `ON CONFLICT` разом із constraints. Upsert дає ідемпотентне перезавантаження, а UNIQUE/PK є арбітром конфлікту. `RETURNING` у data-modifying CTE перетворює DML на audit-зведення без окремих запитів.

**Window functions у ML-проєкті:** дедуплікація подій, «останнє замовлення» для point-in-time ознак, ковзні метрики (`AVG(...) OVER (ROWS ...)`), час між покупками через `LAG`, ранжування для train/test split за часом.

**Якість даних.** `review_id` не унікальний (789 спільних), 547 замовлень мають кілька відгуків, 768 — жодного. 610 товарів без категорії, а більшість текстових відгуків порожні. Статуси суперечать датам (delivered без дати доставки, canceled з датою). Є нічиї в timestamp у 269 покупців.

**`customer_unique_id`.** `customer_id` створюється на кожне замовлення, тож на його рівні повторних покупців 0, а на рівні реального покупця — 2 997 (максимум 17 замовлень). RFM без цього неможливий.

**Грануляність `seller_id × order_id`.** Відгук належить замовленню, а не позиції. Item-level join завищив кількість відгуків на 13 % і змінив gold-статус у 67 продавців. Тому спершу `DISTINCT` пари seller × order, а відгуки окремо зводимо до `order_id`.